# 08 - Error Handling

## 1. Consistent Error Format

**Definition:** Every error from the API should have the **same JSON shape** and the **right status code**, so the frontend can handle all errors in one way.

**Standard shape:**
{"error": {"code": "OUT_OF_STOCK", "message": "Only 2 laptops left", "details": {"available": 2}}}

**Rules:**
- Right status code (400, 404, 409, etc.) plus the same body shape every time
- A machine-readable **code** (for the frontend) and a human **message** (for the user)
- **Never leak stack traces**, SQL, file paths or secrets to the client
- Log the full error on the server with a request id; return only the request id to the client
- Map custom exceptions to status codes in one central place (FastAPI: exception handlers)

**Real-world example:** At checkout, the app gets 409 with code OUT_OF_STOCK and shows "Only 2 left". A crash returns 500 with "Something went wrong, reference id abc123", while the full traceback stays in the server logs.

**Interview line:** "I return a consistent error shape with a code and message, map custom exceptions to status codes centrally, and never leak stack traces; the full error is logged server-side with a request id."

In [ ]:
# ==================================================
# 1. CUSTOM EXCEPTIONS -> ONE CONSISTENT ERROR SHAPE
# ==================================================
import uuid

class AppError(Exception):
    status, code = 400, "BAD_REQUEST"
    def __init__(self, message, details=None):
        self.message, self.details = message, details or {}

class NotFoundError(AppError):
    status, code = 404, "NOT_FOUND"

class OutOfStockError(AppError):
    status, code = 409, "OUT_OF_STOCK"

def error_response(status, code, message, details=None):
    return status, {"error": {"code": code, "message": message, "details": details or {}}}

def handle(route_func):
    """Central handler - like FastAPI exception handlers"""
    try:
        return 200, route_func()
    except AppError as e:
        return error_response(e.status, e.code, e.message, e.details)
    except Exception as e:
        request_id = uuid.uuid4().hex[:6]
        print(f"   [SERVER LOG] request_id={request_id} {type(e).__name__}: {e}")
        return error_response(500, "INTERNAL_ERROR", "Something went wrong", {"request_id": request_id})

stock = {"laptop": 2}

def get_product():
    raise NotFoundError("Product 999 not found")

def place_order():
    if stock["laptop"] < 5:
        raise OutOfStockError("Only 2 laptops left", {"available": stock["laptop"]})

def buggy_route():
    return 10 / 0   # unexpected bug

print("=" * 60)
for name, route in [("GET /products/999", get_product),
                    ("POST /orders qty=5", place_order),
                    ("GET /reports", buggy_route)]:
    status, body = handle(route)
    print(f"[{status}] {name:20} -> {body}")
print("=" * 60)

## Key Takeaways

- Same error shape every time: code, message, details
- Right status code with every error
- Map custom exceptions to status codes in one central handler
- Never leak stack traces; log them server-side with a request id
- Return only the request id to the client on a 500